# 8.2 剪枝 (Pruning)

剪枝通过移除不重要的参数来减少模型大小和计算量。

本节涵盖：
- 非结构化剪枝（幅度剪枝、迭代剪枝）
- 结构化剪枝（通道/头/层剪枝）
- 彩票假设与Lottery Ticket Hypothesis
- Movement Pruning
- LLM剪枝（SparseGPT、ShortGPT、SliceGPT）
- 剪枝后微调恢复

## 1. 非结构化剪枝

### 幅度剪枝 (Magnitude Pruning)
将绝对值最小的权重置零，创建稀疏矩阵。

### 一次性 vs 迭代剪枝
- **一次性**：直接剪到目标稀疏度，精度损失大
- **迭代**：逐步剪枝+微调，精度恢复更好

### 稀疏度与精度
- 70%以下稀疏度：通常精度损失可接受
- 70-90%：需要迭代剪枝+微调
- 90%+：通常需要特殊方法（如彩票假设）

**局限**：非结构化稀疏需要专用稀疏硬件/软件才能实际加速

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

def magnitude_prune(tensor, sparsity=0.5):
    abs_tensor = tensor.abs()
    threshold = torch.quantile(abs_tensor.flatten(), sparsity)
    mask = (abs_tensor > threshold).float()
    pruned = tensor * mask
    return pruned, mask

def iterative_magnitude_prune(model, train_fn, target_sparsity=0.8, n_iterations=5):
    sparsity_per_iter = 1 - (1 - target_sparsity) ** (1 / n_iterations)
    masks = {}
    for iteration in range(n_iterations):
        for name, param in model.named_parameters():
            if param.dim() >= 2:
                threshold = torch.quantile(param.data.abs().flatten(), sparsity_per_iter)
                new_mask = (param.data.abs() > threshold).float()
                if name in masks:
                    masks[name] = masks[name] * new_mask
                else:
                    masks[name] = new_mask
                param.data *= new_mask
        train_fn(model)
        for name, param in model.named_parameters():
            if name in masks:
                param.data *= masks[name]
    return masks

W = torch.randn(256, 256)

print('=== Unstructured Pruning ===')
print(f'Original non-zero: {(W != 0).sum().item():,} / {W.numel():,}')

for sparsity in [0.3, 0.5, 0.7, 0.9]:
    W_p, mask = magnitude_prune(W, sparsity)
    error = (W - W_p).norm() / W.norm()
    actual_sp = (W_p == 0).float().mean()
    print(f'  Sparsity {sparsity:.0%}: actual={actual_sp:.2%}, relative error = {error:.4f}')

model = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10))

def simple_train(m):
    x = torch.randn(16, 64)
    y = torch.randint(0, 10, (16,))
    out = m(x)
    loss = F.cross_entropy(out, y)
    loss.backward()
    with torch.no_grad():
        for p in m.parameters():
            if p.grad is not None:
                p.data -= 0.01 * p.grad
                p.grad = None

masks = iterative_magnitude_prune(model, simple_train, target_sparsity=0.7, n_iterations=3)
total_params = sum(p.numel() for p in model.parameters())
zero_params = sum((p == 0).sum().item() for p in model.parameters())
print(f'\nIterative pruning result: {zero_params/total_params:.1%} sparsity')
print(f'\nKey: Iterative prune+finetune preserves accuracy better than one-shot pruning.')
print(f'Non-structured pruning needs sparse kernels for actual speedup.')

## 2. 结构化剪枝

移除整个结构单元（通道、注意力头、层），产生规则的小矩阵。

### 剪枝类型
| 类型 | 移除对象 | 加速效果 | 精度影响 |
|------|---------|---------|----------|
| 通道剪枝 | 整个输出通道 | 直接 | 中 |
| 头剪枝 | 整个注意力头 | 直接 | 中 |
| 层剪枝 | 整个Transformer层 | 显著 | 大 |
| FFN剪枝 | FFN中间维度 | 直接 | 中 |

### 重要性评估方法
- **L1范数**：通道权重绝对值之和
- **L2范数**：通道权重平方和的根
- **几何中值**：最接近其他通道的通道最不重要
- **梯度信息**：对损失影响最小的通道

**优势**：直接减少矩阵维度，任何硬件都能加速

In [ ]:
class ChannelPruner:
    def __init__(self, model, prune_ratio=0.3):
        self.model = model
        self.prune_ratio = prune_ratio
        self.importance_scores = {}

    def compute_importance(self, x):
        hooks = []
        activations = {}
        def hook_fn(name):
            def fn(module, input, output):
                activations[name] = input[0].detach()
            return fn
        for name, module in self.model.named_modules():
            if isinstance(module, nn.Linear):
                hooks.append(module.register_forward_hook(hook_fn(name)))
        with torch.no_grad():
            self.model(x)
        for h in hooks:
            h.remove()
        for name, param in self.model.named_parameters():
            if 'weight' in name and param.dim() >= 2:
                l1_score = param.data.abs().sum(dim=0)
                self.importance_scores[name] = l1_score
        return self.importance_scores

    def get_prune_mask(self, name, score):
        n_prune = int(len(score) * self.prune_ratio)
        _, indices = score.sort()
        mask = torch.ones_like(score, dtype=torch.bool)
        mask[indices[:n_prune]] = False
        return mask

class StructuredPrunedModel(nn.Module):
    def __init__(self, d_model=256, d_ff=1024, n_heads=8, prune_heads=3):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.keep_heads = n_heads - prune_heads
        self.keep_dim = self.keep_heads * self.d_head
        self.qkv = nn.Linear(d_model, 3 * self.keep_dim)
        self.proj = nn.Linear(self.keep_dim, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.keep_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) / (self.d_head ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, self.keep_dim)
        return self.proj(out) + self.ffn(x)

full_model = nn.TransformerEncoderLayer(256, 8, 1024, batch_first=True)
pruned_model = StructuredPrunedModel(256, 1024, 8, prune_heads=3)

full_params = sum(p.numel() for p in full_model.parameters())
pruned_params = sum(p.numel() for p in pruned_model.parameters())

x = torch.randn(2, 16, 256)
out = pruned_model(x)

print('=== Structured Pruning ===')
print(f'Full model: {full_params:,} params (8 heads)')
print(f'Pruned model: {pruned_params:,} params (5 heads, 3 pruned)')
print(f'Parameter reduction: {(1 - pruned_params/full_params):.1%}')
print(f'Output shape: {out.shape}')

simple_model = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10))
pruner = ChannelPruner(simple_model, prune_ratio=0.3)
x_test = torch.randn(8, 64)
scores = pruner.compute_importance(x_test)
print(f'\nChannel importance scores computed for {len(scores)} layers')
for name, score in scores.items():
    mask = pruner.get_prune_mask(name, score)
    print(f'  {name}: {mask.sum().item()}/{len(mask)} channels kept')

print(f'\nKey: Structured pruning removes entire channels/heads -> direct speedup.')
print(f'Importance scores (L1, gradient-based) determine which channels to prune.')

## 3. 彩票假设与Movement Pruning

### 彩票假设 (Lottery Ticket Hypothesis)
**核心发现**：密集网络中存在稀疏子网络（"中奖彩票"），单独训练这些子网络可达到原网络精度。

### Movement Pruning
传统幅度剪枝在微调时固定mask，Movement Pruning让mask随训练动态调整：
- 权重向零移动的会被剪掉
- 权重远离零的会被保留
- 比幅度剪枝在微调场景下效果更好

### 对比
| 方法 | Mask策略 | 适用场景 | 精度 |
|------|---------|---------|------|
| 幅度剪枝 | 固定（训练前决定） | 一次性剪枝 | 中 |
| 迭代剪枝 | 逐步固定 | 精度敏感 | 高 |
| Movement | 动态（训练中调整） | 微调剪枝 | 高 |
| 彩票假设 | 固定（重新初始化） | 理论研究 | 高 |

In [ ]:
class MovementPruning(nn.Module):
    def __init__(self, in_features, out_features, sparsity=0.5):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.score = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
        self.sparsity = sparsity
        self.bias = nn.Parameter(torch.zeros(out_features))

    def get_mask(self):
        threshold = torch.quantile(self.score.abs().flatten(), self.sparsity)
        return (self.score.abs() > threshold).float()

    def forward(self, x):
        mask = self.get_mask()
        return F.linear(x, self.weight * mask, self.bias)

class LotteryTicketExperiment:
    def __init__(self, d=64, n_classes=10, sparsity=0.8):
        self.d = d
        self.sparsity = sparsity
        self.model = nn.Sequential(
            nn.Linear(d, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, n_classes)
        )
        self.original_init = {n: p.clone() for n, p in self.model.named_parameters()}

    def find_winning_ticket(self, x, y, n_iterations=3):
        results = []
        for iteration in range(n_iterations):
            optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-3)
            for _ in range(20):
                out = self.model(x)
                loss = F.cross_entropy(out, y)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            masks = {}
            sparsity_per_iter = 1 - (1 - self.sparsity) ** (1 / n_iterations)
            for name, param in self.model.named_parameters():
                if param.dim() >= 2:
                    threshold = torch.quantile(param.data.abs().flatten(), sparsity_per_iter)
                    masks[name] = (param.data.abs() > threshold).float()
                    param.data *= masks[name]
            for name, param in self.model.named_parameters():
                if name in self.original_init:
                    param.data = self.original_init[name] * masks.get(name, 1.0)
            with torch.no_grad():
                acc = (self.model(x).argmax(1) == y).float().mean()
            results.append({'iteration': iteration, 'sparsity': sparsity_per_iter, 'acc': acc.item()})
        return results

mp_layer = MovementPruning(64, 32, sparsity=0.5)
x = torch.randn(4, 64)
out = mp_layer(x)
mask = mp_layer.get_mask()

print('=== Movement Pruning ===')
print(f'Weight shape: {mp_layer.weight.shape}')
print(f'Mask sparsity: {(mask == 0).float().mean():.1%}')
print(f'Score gradient flows -> mask adapts during training')

lth = LotteryTicketExperiment(d=64, n_classes=10, sparsity=0.7)
x = torch.randn(16, 64)
y = torch.randint(0, 10, (16,))
results = lth.find_winning_ticket(x, y, n_iterations=3)

print(f'\n=== Lottery Ticket Hypothesis ===')
for r in results:
    print(f'  Iteration {r["iteration"]}: sparsity={r["sparsity"]:.1%}, acc={r["acc"]:.2%}')
print(f'\nKey: Movement pruning adapts masks during training (better for fine-tuning).')
print(f'Lottery ticket: re-initialize to original weights after pruning, then retrain.')

## 4. LLM剪枝

### SparseGPT
**SparseGPT** (Frantar & Alistarh, 2023)：将GPTQ思想扩展到稀疏化，一次性将LLM剪枝到50-60%稀疏度。

### ShortGPT
移除冗余Transformer层。发现深层存在大量功能冗余。

### SliceGPT
通过SVD分解移除主成分的次要分量，在每层后截断最小奇异值对应的维度。

### LLM-Pruner
基于依赖关系的结构化剪枝，分析层间依赖图，确保剪枝不破坏计算图。

### 挑战
- 剪枝后通常需要LoRA微调恢复性能
- 20-30%结构化剪枝后性能显著下降
- 非结构化剪枝（SparseGPT）可达50%+但需稀疏推理支持
- 量化+剪枝组合是前沿方向

In [ ]:
class ShortGPTDemo(nn.Module):
    def __init__(self, d_model=128, n_heads=2, n_layers=6, keep_layers=4):
        super().__init__()
        all_layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model, n_heads, d_model*4, batch_first=True)
            for _ in range(n_layers)
        ])
        self.layers = nn.ModuleList([all_layers[i] for i in range(keep_layers)])
        self.n_original = n_layers
        self.n_kept = keep_layers

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

def layer_redundancy_score(model, x, layer_idx):
    h = x.clone()
    for i in range(layer_idx):
        h = model.layers[i](h)
    h_before = h.clone()
    h_after = model.layers[layer_idx](h_before)
    similarity = torch.cosine_similarity(h_before.flatten(), h_after.flatten(), dim=0)
    return similarity.item()

def sparse_gpt_simplify(W, X, sparsity=0.5, block_size=128):
    d_row, d_col = W.shape
    H = 2 * X @ X.T + 1e-6 * torch.eye(d_col, device=W.device)
    H_inv = torch.linalg.inv(H)
    W_sparse = W.clone()
    n_prune = int(d_row * d_col * sparsity)
    flat = W_sparse.abs().flatten()
    threshold = torch.kthvalue(flat, n_prune + 1).values
    mask = (W_sparse.abs() > threshold).float()
    for j in range(d_col):
        col_mask = mask[:, j]
        pruned_rows = (col_mask == 0).nonzero(as_tuple=True)[0]
        if len(pruned_rows) > 0:
            for row in pruned_rows:
                err = W_sparse[row, j]
                d = H_inv[j, j]
                W_sparse[row, j:] -= err * H_inv[j, j:] / d
                W_sparse[row, j] = 0
    return W_sparse, mask

full_model = ShortGPTDemo(128, 2, 6, 6)
x = torch.randn(2, 16, 128)

print('=== LLM Pruning ===')
print(f'\nShortGPT layer redundancy:')
scores = []
for i in range(min(4, len(full_model.layers))):
    score = layer_redundancy_score(full_model, x, i)
    scores.append((i, score))
    print(f'  Layer {i}: cosine_sim={score:.4f}')

most_redundant = max(scores, key=lambda x: x[1])
print(f'  -> Layer {most_redundant[0]} is most redundant (highest similarity)')

W = torch.randn(64, 128)
X = torch.randn(128, 16)
W_sparse, mask = sparse_gpt_simplify(W, X, sparsity=0.5)
sparse_error = (W - W_sparse).norm() / W.norm()
actual_sp = (W_sparse == 0).float().mean()

print(f'\nSparseGPT-style pruning:')
print(f'  Target sparsity: 50%')
print(f'  Actual sparsity: {actual_sp:.1%}')
print(f'  Relative error: {sparse_error:.4f}')

print(f'\nKey: SparseGPT uses Hessian info to compensate pruning error (like GPTQ for sparsity).')
print(f'ShortGPT removes redundant layers based on cosine similarity.')
print(f'After pruning, LoRA fine-tuning is typically needed to recover performance.')

## 5. 剪枝总结

| 方法 | 类型 | 稀疏度上限 | 硬件加速 | 精度影响 |
|------|------|-----------|---------|----------|
| 幅度剪枝 | 非结构化 | 90%+ | 需稀疏支持 | 低 |
| 迭代剪枝 | 非结构化 | 90%+ | 需稀疏支持 | 低 |
| Movement | 非结构化 | 90%+ | 需稀疏支持 | 低 |
| 通道剪枝 | 结构化 | 50% | 直接加速 | 中 |
| 头剪枝 | 结构化 | 50% | 直接加速 | 中 |
| 层剪枝 | 结构化 | 30% | 显著加速 | 大 |
| SparseGPT | 非结构化 | 60% | 需稀疏支持 | 低 |

**实践建议**：
- 需要实际加速 → 结构化剪枝
- 追求极致压缩 → 非结构化 + 稀疏推理引擎
- LLM场景 → SparseGPT或ShortGPT + LoRA微调
- 量化+剪枝组合可进一步压缩